<a href="https://colab.research.google.com/github/Eliezer-Carvalho/Adamastor-GPT/blob/main/Adamastor_GPT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### **Adamastor-GPT**

Este modelo é baseado na arquitetura <b> Transformer </b>, inspirado nos modelos do tipo <b> GPT (<i>Generative Pre-trained Transformer</i>)</b>. <br>

Ao longo deste Colab, iremos navegar pelas componentes cruciais dos modelos GPT-like, que atualmente dominam grande parte do mundo da Inteligência Artificial.

<b> No final, teremos um modelo estilo GPT funcional, capaz de realizar inferência de forma limpa e consistente. </b>

In [174]:
from tokenizers import Tokenizer
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

device = "cuda" if torch.cuda.is_available() else "cpu" #Mover tudo para a GPU #Fulcral para treino

### Dados do Modelo

In [60]:
Sequence_Length = 32
Batch_Size = 16
Embedding_Dimension = 64
Head_Size = 8 #Head_Size = Embedding_Dimension // Number_Head
Number_Head = 8

###


---

### Tokenization


In [69]:
Tokenizer = Tokenizer.from_file ("GAMA")

with open ("Os Lusiadas.txt", "r", encoding = "utf-8") as f:
  dataset = f.read()

Tokenization = Tokenizer.encode (dataset)
print (len(Tokenization.ids))
print (len(Tokenization.tokens))

tensor = torch.tensor (Tokenization.ids, dtype = torch.long)
print (tensor [:50])

n = int (0.9 * (len(tensor)))
dados_treino = tensor [:n] #90% Dados de Treino
dados_teste = tensor [n:] #10% Dados de Teste

75042
75042
tensor([  27,   31,    3,   24,   33,   31,   70,   14,   17,   14,   31,    3,
           0,   24,   58,   83,   95,  692,  120,  541,    0, 2342,   22,    3,
           0,  344,  672, 1062, 2563,  541, 2461,  398,  143,  138, 2854,  115,
         208, 4182,  783, 3295, 2515,  575, 4573,  694,  398, 1999,  662,   39,
         543,  114])


###


---

### Batch e Sequence Length

In [71]:
torch.manual_seed (2000)

def Batch ():
  idx = torch.randint (0, len(dados_treino) - Sequence_Length, (Batch_Size, ))
  x = torch.stack ([dados_treino [i: i + Sequence_Length] for i in idx])
  y = torch.stack ([dados_treino [i + 1: i + Sequence_Length + 1] for i in idx])

  return x, y

x, y = Batch()

x = x.to(device) #Passar para GPU
print (x.shape) #(B, T)
print (x[1])

y = y.to(device) #Passar para GPU
print (y.shape) #(B, T)
print (y[1])

torch.Size([16, 32])
tensor([ 106, 1249, 1065,  875,  554, 3649,  878, 1657, 1544, 1065, 1308,  290,
        2747,  207, 4168,  264,  170,  162,  277,  166,  396,  909, 2662, 3882,
        3426,  233, 1395, 3344, 3492,   94, 2314,  534])
torch.Size([16, 32])
tensor([1249, 1065,  875,  554, 3649,  878, 1657, 1544, 1065, 1308,  290, 2747,
         207, 4168,  264,  170,  162,  277,  166,  396,  909, 2662, 3882, 3426,
         233, 1395, 3344, 3492,   94, 2314,  534, 1092])


###


---



### Definição de uma Head

In [63]:
class Head (nn.Module): #Head = Attention
  def __init__(self):
    super().__init__()
    self.Query = nn.Linear (Embedding_Dimension, Head_Size, bias = False)
    self.Key = nn.Linear (Embedding_Dimension, Head_Size, bias = False)
    self.Value = nn.Linear (Embedding_Dimension, Head_Size, bias = False)

  def forward (self, x):
    B, T, C = x.shape

    Q = self.Query (x)
    K = self.Key (x)
    V = self.Value (x)

    attention_score = Q @ K.transpose (-2, -1) * (Head_Size ** -0.5)

    tril = torch.tril (torch.ones (T, T, device = x.device))
    attention_score = attention_score.masked_fill (tril == 0, float ("-inf"))

    attention_score = F.softmax (attention_score, dim = -1)

    output = attention_score @ V
    return output


###


---


### Definição de Multi Head

In [ ]:
class Multi_Head (nn.Module):
  def __init__(self):
    super().__init__()
    self.Concat = nn.ModuleList ([Head() for i in range (Number_Head)]) #Head Concatenation #Definição de Multi Head Attention
    self.Output_Projection = nn.Linear (Embedding_Dimension, Embedding_Dimension) #Necessário para misturar a informação das Heads #Desta forma volta também à dimensão (B, T, C)

  def forward (self, x):

    output = self.Output_Projection (torch.cat ([h (x) for h in self.Concat], dim = -1))
    return output


###


---


### Definição de um Block (Pre-LayerNorm Transformer)

Em modelos GPT, um Block é:

- Layer Norm
- Attention
- Residual Connection (Add)
- Layer Norm
- Feed Forward Neural Network
- Residual Connection (Add)

In [64]:
class Block (nn.Module):
  def __init__(self):
    super().__init__()

    self.LayerNorm_Attention = nn.LayerNorm (Embedding_Dimension)
    self.Masked_Multi_Head_Attention = Multi_Head()


    self.FeedForward_NeuralNet = nn.Sequential (
        nn.Linear (Embedding_Dimension, 4 * Embedding_Dimension),
        nn.ReLU(),
        nn.Linear (Embedding_Dimension * 4, Embedding_Dimension)
    )
    self.LayerNorm_FNN = nn.LayerNorm (Embedding_Dimension)

  def forward (self, x):
    Norm_Attention = self.Masked_Multi_Head_Attention (self.LayerNorm_Attention (x)) #Norm #LayerNorm
    Add_Attention = x + Norm_Attention #Add #Residual Connection
    x = Add_Attention

    Norm_FNN = self.FeedForward_NeuralNet (self.LayerNorm_FNN (x)) #Norm #LayerNorm
    Add_FNN = x + Norm_FNN #Add #Residual Connection
    x = Add_FNN

    return x


###


---


### Modelo Adamastor-GPT

In [65]:
class ADAMASTOR_GPT (nn.Module):
  def __init__(self):
    super().__init__()
    self.Embedding = nn.Embedding (Tokenizer.get_vocab_size(), Embedding_Dimension)
    self.Positional_Encoding = nn.Embedding (Sequence_Length, Embedding_Dimension)

    self.Blocks = nn.Sequential (
        Block(),
        Block(),
        Block(),
        Block()
    )

    self.Language_Modeling = nn.Linear (Embedding_Dimension, Tokenizer.get_vocab_size()) #Converte o vetor final do modelo em logits sobre o vocabulário inteiro.

  def forward (self, x):
    B, T = x.shape

    x = self.Embedding (x) + self.Positional_Encoding (torch.arange (T, device = x.device))

    x = self.Blocks(x)

    logits = self.Language_Modeling (x)

    return logits #Return de um tensor (4, 8, 5000)



###


---


### Treino do Modelo (Super-Importante)


In [167]:
def training (modelo, steps):

  modelo.train() #Ativa o treino
  optimizer = torch.optim.AdamW (modelo.parameters(), lr = 1e-4) #Usa-se AdamW ou SGD #Responsável por atualizar os pesos do modelo
  loss_function = nn.CrossEntropyLoss() #Função de perda que mede quão diferente está a previsão do modelo da resposta correta.

  for i in range (steps):

    x, y = Batch()
    x, y = x.to(device), y.to(device)

    logits = modelo (x)

    #print (logits.shape) #(16,32,5000)
    B, T, V = logits.shape #(Batch, Sequence Length, Vocab_Size)

    loss =  loss_function(
        logits.view (B * T, V),
        y.view (B * T)
    ) #Entrada que o Cross Entropy espera

    optimizer.zero_grad() #Zera os gradientes acumulados nos parâmetros.
    loss.backward() #Backpropagation

    optimizer.step() #Optimizer aplicado

    if i % 500 == 0: #Printa de 100 em 100
      print(f"Step {i} | Loss Value = {loss.item()} | PPL: {math.exp(loss.item())}")
      #Perplexity calcula quantas opções o modelo acha corretas para o próximo token, ou seja se tiver um perplexity baixo é bom, quer dizer que ele tem sempre a certeza do próximo token, o que significa que ele tem uma loss baixa. Podemos entender como uma maneira diferente de ver a loss

    if i % 1000 == 0:
      print(gerar_texto(modelo, 100))



###


---

### Geração de Texto

In [168]:
def gerar_texto (modelo, max_tokens):

    device = next(modelo.parameters()).device  #Vai buscar o modelo à GPU
    tokens = torch.zeros((1, 1), dtype = torch.long, device = device)
    tokens = tokens.to(device)

    modelo.eval() #Desliga o treino

    with torch.no_grad(): #Desliga o gradiente
      for _ in range(max_tokens):

        contexto_max = tokens[:, -Sequence_Length:] #Alimento o modelo com o máximo de tokens possíveis, não posso dar um contexto maior do que daquele com que treinou

        logits = modelo(contexto_max) #(B,T,VOCAB_SIZE)
        logits = logits[:, -1, :] #O que vem depois do último token que gerou ?

        probabilidade = F.softmax(logits, dim = -1) #Aplica normalização

        next_token = torch.multinomial(probabilidade, num_samples = 1) #Seleciona um token aleatoriamente ponderado

        tokens = torch.cat([tokens, next_token], dim = 1) #Adiciona a uma lista com todos os outros tokens

      text = Tokenizer.decode(tokens[0].tolist()) #Decode do token para palavra ou sub palavra

      print(text)


###


---

### Adamastor-GPT Treino e Inferência

In [161]:
modelo = ADAMASTOR_GPT().to(device)

training (modelo, 1000)

gerar_texto (modelo, 200)

Loss Value = 8.923751831054688
Loss Value = 8.695813179016113
Loss Value = 8.4470853805542
Loss Value = 8.3096342086792
Loss Value = 8.014472007751465
Loss Value = 7.838489532470703
Loss Value = 7.905187129974365
Loss Value = 7.890261173248291
Loss Value = 7.818908214569092
Loss Value = 7.826442718505859

 ouro e  delic es Pel er  bu -m .» Assi  que  eram  sa ando,  tar trat ães  as e  te e  ís o D o mar  do  alegr at E  sempre  apac pensament trat and ira  as  dá  Já  as damas  T aram,  vol fonte  os  nem  a,  A pr cert os mares  estava,  Prom dur ia,  P entre  Ag Su partes  em eita  a mort aper ar  o que  o mer eu  le não  geração  em,  Assi  namor o, A - « fei dos  Lhe  ênci press a  mad que  Se  em s manh ilustre e  e,  virtude  à morte  Se  e  :  D Nair Pera os  aos  bombar ra  pod e  ta e  per eiros,  as  nos  bi os,  e  puder Torn os;  De quem  pela  ores  ca  quem  í ava,  terr el falt desej não  osas;  
 
 gentes  com  as  água  e Que os  em mui  a junto  Tr inclin dos  de m a

In [175]:
print (modelo.state_dict())
for param in modelo.state_dict():
  print (param)

OrderedDict({'Embedding.weight': tensor([[ 1.1122, -0.0848,  0.8114,  ...,  0.6418,  1.4143, -0.2019],
        [ 0.3170, -0.4323,  0.7347,  ...,  1.6249,  0.6717,  1.3599],
        [ 1.5545,  1.5967, -2.9789,  ..., -0.7975, -0.2867,  0.2492],
        ...,
        [-0.3345,  1.7997,  0.3230,  ...,  0.4787, -0.3736, -0.1081],
        [-1.0041,  1.2184,  1.0726,  ...,  0.6672,  1.4308,  0.1529],
        [-1.0450,  0.4027,  1.0616,  ..., -0.8468, -0.0888,  0.6353]]), 'Positional_Encoding.weight': tensor([[-0.5901, -2.3500,  0.7688,  ...,  0.7244,  0.6697, -0.0214],
        [-0.3328,  2.1737, -0.9501,  ..., -1.1450,  0.5946,  0.4687],
        [-0.5025,  0.7172, -0.4471,  ...,  0.3428,  0.2096, -0.3656],
        ...,
        [ 2.2424, -0.6327,  1.2627,  ...,  0.8824, -0.9416, -0.2571],
        [ 1.2098,  0.3003,  0.0382,  ...,  0.5622, -0.5439,  1.0421],
        [ 0.6397,  0.2021, -0.2346,  ...,  2.3276,  0.4930,  0.1550]]), 'Blocks.0.LayerNorm_Attention.weight': tensor([1.0142, 0.9919, 1.00

In [173]:
print (sum(p.numel() for p in modelo.parameters()))
save = torch.save(modelo.state_dict(), "Adamastor-GPT.pth")

O teu ADAMASTOR_GPT é:

um Transformer decoder-only (tipo GPT)
com:
4 camadas (blocos)
8 cabeças de atenção
embeddings de tamanho 64
vocabulário de 5000 tokens
contexto máximo de 32 tokens

FAZER ESTA LEITURA